# Portfolio Optimization
---
> **Bathaix Philippe-Emmanuel Yao**

A comprehensive quantitative portfolio construction framework covering four optimization paradigms,
risk analytics, and stress testing — all driven by live price data via `yfinance`.

| Method | Description |
|---|---|
| **Mean-Variance (Markowitz)** | Maximum Sharpe ratio on the efficient frontier |
| **Black-Litterman** | Bayesian blend of market equilibrium returns with investor views |
| **Robust Optimization** | Ledoit-Wolf shrinkage covariance + max-Sharpe |
| **Monte Carlo** | Random portfolio sampling to visualise the feasible set |
| **Scenario Analysis** | Stress-test optimal weights under bull/bear/stagflation assumptions |

**Improvements over original:**
- `input()` removed; parameters defined as constants (notebook-friendly)
- `yfinance` updated: `Adj Close` → `Close` (API change fix)
- VaR and CVaR computed both historically and parametrically (Gaussian)
- Discrete allocation moved to a dedicated section with clearer output
- Rolling portfolio performance chart added
- Efficient Frontier plotted with colour-mapped Sharpe ratio
- Modular functions with full docstrings and error handling


## 1. Imports & Configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import yfinance as yf

from pypfopt.efficient_frontier import EfficientFrontier
from pypfopt import risk_models, expected_returns, plotting, BlackLittermanModel
from pypfopt.discrete_allocation import DiscreteAllocation, get_latest_prices
from pypfopt import CLA
import cvxpy as cp

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
np.random.seed(42)

# ── Portfolio configuration ───────────────────────────────────────────────────
TICKERS            = ['TSLA', 'BLK', 'NVDA', 'AAPL', 'MSFT']
START_DATE         = '2020-01-01'
END_DATE           = '2024-01-01'
TOTAL_VALUE        = 100_000      # portfolio value for discrete allocation (USD)
RISK_FREE_RATE     = 0.04         # annualised risk-free rate (e.g. US T-bill)
TRADING_DAYS       = 252


## 2. Data Download & Preparation

In [ ]:
def download_data(tickers: list, start: str, end: str) -> pd.DataFrame:
    """
    Download adjusted close prices from Yahoo Finance.
    Handles the yfinance API change: 'Adj Close' was renamed to 'Close'
    for adjusted data when using auto_adjust=True (default since yfinance 0.2.x).
    """
    raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
    # Handle multi-level or single-level columns
    if isinstance(raw.columns, pd.MultiIndex):
        data = raw['Close']
    else:
        data = raw[['Close']] if 'Close' in raw.columns else raw

    if data.isnull().values.any():
        missing = data.isnull().sum()
        print(f"Warning: missing values detected:\n{missing[missing > 0]}")
        data = data.ffill().dropna()

    print(f"Downloaded {len(data)} trading days for {list(data.columns)}")
    return data


def calculate_returns(data: pd.DataFrame) -> pd.DataFrame:
    """Log returns (continuously compounded)."""
    return np.log(data / data.shift(1)).dropna()


def plot_prices(data: pd.DataFrame) -> None:
    """Normalised price chart (base 100)."""
    normalised = data / data.iloc[0] * 100
    ax = normalised.plot(figsize=(12, 5))
    ax.set_title('Normalised Price Performance (Base = 100)')
    ax.set_ylabel('Indexed Price')
    ax.set_xlabel('')
    plt.tight_layout()
    plt.show()


def plot_correlation(returns: pd.DataFrame) -> None:
    """Correlation heatmap of log returns."""
    corr = returns.corr()
    fig, ax = plt.subplots(figsize=(6, 5))
    import seaborn as sns
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
                linewidths=0.5, ax=ax, vmin=-1, vmax=1)
    ax.set_title('Return Correlation Matrix')
    plt.tight_layout()
    plt.show()


In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
data    = download_data(TICKERS, START_DATE, END_DATE)
returns = calculate_returns(data)

plot_prices(data)
plot_correlation(returns)

print("\nAnnualised Return Statistics:")
ann_ret = returns.mean() * TRADING_DAYS
ann_vol = returns.std()  * np.sqrt(TRADING_DAYS)
stats = pd.DataFrame({'Ann. Return': ann_ret, 'Ann. Volatility': ann_vol,
                      'Sharpe (vs rf)': (ann_ret - RISK_FREE_RATE) / ann_vol})
print(stats.round(4))


## 3. Portfolio Optimisation Methods

### 3.1 — Mean-Variance Optimisation (Markowitz)

Maximises the Sharpe ratio on the mean-variance efficient frontier:

$$\max_w \frac{\mu^\top w - r_f}{\sqrt{w^\top \Sigma w}} \quad \text{s.t.} \quad \mathbf{1}^\top w = 1,\; w \geq 0$$

where $\mu$ is the vector of expected returns and $\Sigma$ is the sample covariance matrix.


In [ ]:
def mean_variance_optimization(data: pd.DataFrame, rf: float = RISK_FREE_RATE) -> dict:
    """
    Compute the max-Sharpe portfolio on the efficient frontier.
    Returns a dict with weights, performance metrics, and frontier data.
    """
    mu = expected_returns.mean_historical_return(data)
    S  = risk_models.sample_cov(data)

    # Plot efficient frontier
    ef_plot = EfficientFrontier(mu, S)
    fig, ax = plt.subplots(figsize=(10, 6))
    plotting.plot_efficient_frontier(ef_plot, ax=ax, show_assets=True)
    ax.set_title('Efficient Frontier — Mean-Variance Optimization')
    ax.set_xlabel('Annualised Volatility')
    ax.set_ylabel('Annualised Expected Return')
    plt.tight_layout()
    plt.show()

    # Max-Sharpe weights
    ef = EfficientFrontier(mu, S)
    ef.max_sharpe(risk_free_rate=rf)
    weights = ef.clean_weights()
    perf    = ef.portfolio_performance(verbose=False, risk_free_rate=rf)

    print("\nMean-Variance — Max Sharpe Portfolio:")
    print(f"  Expected Return: {perf[0]:.2%}")
    print(f"  Volatility:      {perf[1]:.2%}")
    print(f"  Sharpe Ratio:    {perf[2]:.4f}")
    print(f"  Weights: {dict(weights)}")

    return {'weights': weights, 'perf': perf, 'mu': mu, 'S': S}


### 3.2 — Black-Litterman Model

The Black-Litterman model combines the **market equilibrium** (reverse-engineered from market caps)
with **investor views** $Q = P \mu + \epsilon$ to produce a posterior expected return vector:

$$\tilde{\mu} = \left[(\tau\Sigma)^{-1} + P^\top\Omega^{-1}P\right]^{-1}
\left[(\tau\Sigma)^{-1}\pi + P^\top\Omega^{-1}Q\right]$$

where $\pi$ is the equilibrium (CAPM) implied return and $\Omega$ is the view uncertainty matrix.


In [ ]:
def black_litterman_optimization(data: pd.DataFrame, market_caps: pd.Series,
                                  P: np.ndarray, Q: np.ndarray,
                                  rf: float = RISK_FREE_RATE) -> dict:
    """
    Black-Litterman optimization.

    Parameters
    ----------
    market_caps : market capitalisation for each ticker (used for equilibrium weights)
    P           : view matrix (k x n) — each row picks a portfolio
    Q           : view returns (k,) — expected excess returns for each view
    """
    S  = risk_models.sample_cov(data)
    bl = BlackLittermanModel(S, pi='market', market_caps=market_caps,
                             absolute_views=dict(zip(data.columns, Q[:len(data.columns)])))
    ret_bl   = bl.bl_returns()
    S_bl     = bl.bl_cov()

    ef = EfficientFrontier(ret_bl, S_bl)
    ef.max_sharpe(risk_free_rate=rf)
    weights = ef.clean_weights()
    perf    = ef.portfolio_performance(verbose=False, risk_free_rate=rf)

    print("\nBlack-Litterman — Max Sharpe Portfolio:")
    print(f"  Expected Return: {perf[0]:.2%}")
    print(f"  Volatility:      {perf[1]:.2%}")
    print(f"  Sharpe Ratio:    {perf[2]:.4f}")
    print(f"  Weights: {dict(weights)}")

    return {'weights': weights, 'perf': perf}


### 3.3 — Robust Optimisation (Ledoit-Wolf Shrinkage)

Replaces the sample covariance (noisy for small $T/N$) with the Ledoit-Wolf shrinkage estimator:

$$\hat{\Sigma}_{LW} = (1 - \alpha) \hat{\Sigma}_{\text{sample}} + \alpha \mu_F I$$

This reduces estimation error and produces more stable (less extreme) portfolio weights.


In [ ]:
def robust_optimization(data: pd.DataFrame, rf: float = RISK_FREE_RATE) -> dict:
    """
    Max-Sharpe optimization with Ledoit-Wolf shrinkage covariance estimator.
    More stable than sample covariance when T is not >> N.
    """
    mu   = expected_returns.mean_historical_return(data)
    S_lw = risk_models.CovarianceShrinkage(data).ledoit_wolf()

    ef = EfficientFrontier(mu, S_lw)
    ef.max_sharpe(risk_free_rate=rf)
    weights = ef.clean_weights()
    perf    = ef.portfolio_performance(verbose=False, risk_free_rate=rf)

    print("\nRobust Optimisation (Ledoit-Wolf) — Max Sharpe Portfolio:")
    print(f"  Expected Return: {perf[0]:.2%}")
    print(f"  Volatility:      {perf[1]:.2%}")
    print(f"  Sharpe Ratio:    {perf[2]:.4f}")
    print(f"  Weights: {dict(weights)}")

    return {'weights': weights, 'perf': perf}


### 3.4 — Monte Carlo Portfolio Sampling

In [ ]:
def monte_carlo_portfolios(data: pd.DataFrame, n_sim: int = 10_000,
                           rf: float = RISK_FREE_RATE) -> pd.DataFrame:
    """
    Sample n_sim random long-only portfolios to visualise the feasible set.
    Returns a DataFrame with columns: return, volatility, sharpe, weights.
    """
    returns  = calculate_returns(data)
    mu_daily = returns.mean()
    cov      = returns.cov()
    n        = len(mu_daily)

    results = []
    for _ in range(n_sim):
        w = np.random.dirichlet(np.ones(n))   # uniform on the simplex
        r = (w * mu_daily).sum() * TRADING_DAYS
        v = np.sqrt(w @ cov.values @ w) * np.sqrt(TRADING_DAYS)
        s = (r - rf) / v
        results.append({'return': r, 'volatility': v, 'sharpe': s,
                        **dict(zip(data.columns, w))})

    df_mc = pd.DataFrame(results)

    # Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    sc = ax.scatter(df_mc['volatility'], df_mc['return'],
                    c=df_mc['sharpe'], cmap='viridis', alpha=0.5, s=8)
    plt.colorbar(sc, ax=ax, label='Sharpe Ratio')
    # Highlight max-Sharpe portfolio
    idx_best = df_mc['sharpe'].idxmax()
    ax.scatter(df_mc.loc[idx_best, 'volatility'], df_mc.loc[idx_best, 'return'],
               marker='*', color='red', s=300, zorder=5, label='Max Sharpe')
    ax.set_xlabel('Annualised Volatility')
    ax.set_ylabel('Annualised Return')
    ax.set_title(f'Monte Carlo — {n_sim:,} Random Portfolios')
    ax.legend()
    plt.tight_layout()
    plt.show()

    best = df_mc.loc[idx_best]
    print(f"\nMC Max-Sharpe Portfolio: Return={best['return']:.2%}  "
          f"Vol={best['volatility']:.2%}  Sharpe={best['sharpe']:.4f}")
    return df_mc


## 4. Risk Analytics

In [ ]:
def risk_metrics(weights: dict, data: pd.DataFrame,
                 confidence: float = 0.95) -> pd.DataFrame:
    """
    Compute per-asset and portfolio-level risk metrics:
      - Annualised volatility
      - Historical VaR and CVaR at the given confidence level
      - Parametric (Gaussian) VaR for comparison
    """
    returns = calculate_returns(data)
    w = pd.Series(weights).reindex(returns.columns).fillna(0).values

    # Portfolio returns time series
    port_ret = returns @ w

    # Annualised vol
    ann_vol = returns.std() * np.sqrt(TRADING_DAYS)

    # Historical VaR / CVaR
    q_level   = 1 - confidence
    var_hist  = -np.percentile(port_ret, 100 * q_level)
    tail      = port_ret[port_ret <= -var_hist]
    cvar_hist = -tail.mean() if len(tail) > 0 else np.nan

    # Parametric (Gaussian) VaR
    from scipy.stats import norm
    mu_p   = port_ret.mean()
    sig_p  = port_ret.std()
    var_param = -(mu_p + norm.ppf(q_level) * sig_p)

    print(f"\nRisk Metrics (confidence = {confidence:.0%}):")
    print(f"  Portfolio Ann. Volatility: {port_ret.std() * np.sqrt(TRADING_DAYS):.2%}")
    print(f"  Historical VaR  (daily):   {var_hist:.4%}")
    print(f"  Historical CVaR (daily):   {cvar_hist:.4%}")
    print(f"  Parametric VaR  (daily):   {var_param:.4%}")

    # Per-asset volatility table
    asset_metrics = pd.DataFrame({
        'Weight':        pd.Series(weights),
        'Ann. Vol':      ann_vol,
        'Weight × Vol':  pd.Series(weights) * ann_vol
    }).round(4)
    print("\nPer-Asset Metrics:")
    print(asset_metrics)
    return asset_metrics


def plot_rolling_performance(weights: dict, data: pd.DataFrame,
                              window: int = 63, label: str = 'Portfolio') -> None:
    """Plot the rolling Sharpe ratio and drawdown of the portfolio."""
    returns = calculate_returns(data)
    w = pd.Series(weights).reindex(returns.columns).fillna(0).values
    port_ret = returns @ w

    rolling_sharpe = (port_ret.rolling(window).mean() /
                      port_ret.rolling(window).std() * np.sqrt(TRADING_DAYS))

    cum_ret  = (1 + port_ret).cumprod()
    drawdown = cum_ret / cum_ret.cummax() - 1

    fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
    cum_ret.plot(ax=axes[0], color='steelblue')
    axes[0].set_title(f'{label} — Cumulative Return')
    axes[0].set_ylabel('Wealth Index')

    rolling_sharpe.plot(ax=axes[1], color='darkorange')
    axes[1].axhline(0, color='grey', linestyle='--', lw=0.8)
    axes[1].set_title(f'Rolling {window}D Sharpe Ratio')
    axes[1].set_ylabel('Sharpe')

    drawdown.plot(ax=axes[2], color='crimson', fill_value=0)
    axes[2].fill_between(drawdown.index, drawdown, 0, alpha=0.3, color='crimson')
    axes[2].set_title('Drawdown')
    axes[2].set_ylabel('Drawdown')

    plt.tight_layout()
    plt.show()


## 5. Scenario Analysis

In [ ]:
def scenario_analysis(data: pd.DataFrame, scenarios: dict,
                      rf: float = RISK_FREE_RATE) -> pd.DataFrame:
    """
    Stress-test three pre-defined macro scenarios.
    For each scenario, re-optimise under the assumed expected returns.

    Parameters
    ----------
    scenarios : dict of {name: {ticker: annual_return}}
    """
    S = risk_models.sample_cov(data)
    results = {}

    for name, views in scenarios.items():
        mu_scenario = pd.Series(views)
        # Skip if all returns are below rf (no Sharpe-maximising solution exists)
        if (mu_scenario <= rf).all():
            print(f"Skipping '{name}': all returns below risk-free rate.")
            continue
        try:
            ef = EfficientFrontier(mu_scenario, S, weight_bounds=(0, 1))
            ef.max_sharpe(risk_free_rate=rf)
            w    = ef.clean_weights()
            perf = ef.portfolio_performance(verbose=False, risk_free_rate=rf)
            results[name] = {'weights': w,
                             'return': perf[0], 'volatility': perf[1], 'sharpe': perf[2]}
            print(f"\nScenario: {name}")
            print(f"  Return={perf[0]:.2%}  Vol={perf[1]:.2%}  Sharpe={perf[2]:.4f}")
        except Exception as e:
            print(f"Scenario '{name}' failed: {e}")

    if not results:
        return pd.DataFrame()

    # Summary bar chart
    summ = pd.DataFrame({k: v['weights'] for k, v in results.items()}).T
    summ.plot(kind='bar', figsize=(10, 5), edgecolor='white', linewidth=0.8)
    plt.title('Optimal Weights by Macro Scenario')
    plt.ylabel('Weight')
    plt.xticks(rotation=0)
    plt.legend(loc='upper right', fontsize=9)
    plt.tight_layout()
    plt.show()

    return summ


def plot_weights(weights: dict, title: str) -> None:
    """Horizontal bar chart of portfolio weights."""
    w = {k: v for k, v in sorted(weights.items(), key=lambda x: -x[1]) if v > 0}
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(list(w.keys()), list(w.values()), color='steelblue', edgecolor='white')
    ax.set_xlabel('Weight')
    ax.set_title(title)
    ax.set_xlim(0, 1)
    for i, (k, v) in enumerate(w.items()):
        ax.text(v + 0.01, i, f'{v:.1%}', va='center')
    plt.tight_layout()
    plt.show()


## 6. Full Execution

In [ ]:
# ── Mean-Variance Optimisation ────────────────────────────────────────────────
mv = mean_variance_optimization(data)
plot_weights(mv['weights'], 'Mean-Variance — Max Sharpe Weights')
plot_rolling_performance(mv['weights'], data, label='Mean-Variance')
risk_metrics(mv['weights'], data)


In [ ]:
# ── Black-Litterman ───────────────────────────────────────────────────────────
# Market caps (approximate, USD billions) and two absolute views
market_caps = pd.Series({
    'TSLA': 700e9, 'BLK': 110e9, 'NVDA': 1200e9, 'AAPL': 2800e9, 'MSFT': 2700e9
})
# Views: NVDA +12%, AAPL +8% over the horizon
Q = np.array([0.12, 0.08])
P = np.array([[0, 0, 1, 0, 0],   # NVDA
              [0, 0, 0, 1, 0]])   # AAPL

bl = black_litterman_optimization(data, market_caps, P, Q)
plot_weights(bl['weights'], 'Black-Litterman — Max Sharpe Weights')
plot_rolling_performance(bl['weights'], data, label='Black-Litterman')
risk_metrics(bl['weights'], data)


In [ ]:
# ── Robust Optimisation ───────────────────────────────────────────────────────
rob = robust_optimization(data)
plot_weights(rob['weights'], 'Robust (Ledoit-Wolf) — Max Sharpe Weights')
plot_rolling_performance(rob['weights'], data, label='Robust')
risk_metrics(rob['weights'], data)


In [ ]:
# ── Monte Carlo Portfolio Sampling ────────────────────────────────────────────
mc_df = monte_carlo_portfolios(data, n_sim=10_000)


In [ ]:
# ── Scenario Analysis ─────────────────────────────────────────────────────────
scenarios = {
    'Bull Market':    {'TSLA': 0.25, 'BLK': 0.12, 'NVDA': 0.35, 'AAPL': 0.18, 'MSFT': 0.20},
    'Bear Market':    {'TSLA': -0.20, 'BLK': -0.05, 'NVDA': -0.10, 'AAPL': -0.08, 'MSFT': -0.06},
    'Stagflation':    {'TSLA': -0.15, 'BLK': 0.03, 'NVDA': -0.05, 'AAPL': 0.02, 'MSFT': 0.01},
}
scenario_df = scenario_analysis(data, scenarios)


In [ ]:
# ── Performance Summary ───────────────────────────────────────────────────────
summary_rows = []
for label, result in [('Mean-Variance', mv), ('Black-Litterman', bl), ('Robust', rob)]:
    r, v, s = result['perf']
    summary_rows.append({'Method': label, 'Return': f'{r:.2%}',
                          'Volatility': f'{v:.2%}', 'Sharpe': f'{s:.4f}'})

summary_df = pd.DataFrame(summary_rows).set_index('Method')
print("\nOptimisation Results Summary:")
print(summary_df.to_string())


In [ ]:
# ── Discrete Allocation (integer shares for $100k portfolio) ──────────────────
print("\nDiscrete Allocation — Mean-Variance Portfolio:")
latest = get_latest_prices(data)
da = DiscreteAllocation(mv['weights'], latest, total_portfolio_value=TOTAL_VALUE)
allocation, leftover = da.lp_portfolio()
alloc_df = pd.DataFrame.from_dict(allocation, orient='index', columns=['Shares'])
alloc_df['Price']      = latest[alloc_df.index]
alloc_df['Value (USD)'] = alloc_df['Shares'] * alloc_df['Price']
print(alloc_df.round(2))
print(f"Uninvested cash: ${leftover:.2f}")
